In [6]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Updated import to match the CRUD Python module file name from Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Updated username and password to match the aacuser credentials
# set up in the Module Three milestone

username = "aacuser"
password = "Gary.Travis"

# Connect to database via CRUD Module
# Renamed variable from db to shelter to match Project One naming convention
shelter = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(shelter.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will return a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Updated image filename to the Grazioso Salvare logo provided in the Codio code_files directory
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),

    # Added the Grazioso Salvare logo image centered at the top of the dashboard
    html.Center(
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()),
            style={'height': '150px'}
        )
    ),

    # Updated dashboard title and added unique identifier as required by the assignment
    html.Center(html.B(html.H1('Grazioso Salvare Rescue Animal Dashboard'))),
    html.Center(html.H4('Gary Travis - CS340')),
    html.Hr(),

    html.Div(
        # Added radio buttons for the interactive filtering options
        # Options match the rescue types from the Dashboard Specifications Document
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'Water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'Wilderness'},
                {'label': 'Disaster or Individual Tracking', 'value': 'Disaster'},
                {'label': 'Reset (Show All)', 'value': 'Reset'}
            ],
            value='Reset',
            labelStyle={'display': 'inline-block', 'marginRight': '20px'},
            style={'padding': '10px'}
        )
    ),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[
            {"name": i, "id": i, "deletable": False, "selectable": True}
            for i in df.columns
        ],
        data=df.to_dict('records'),
        # Set page_size to limit rows displayed at once for easier client navigation
        page_size=10,
        # Enabled native sorting so the client can click column headers to sort
        sort_action='native',
        # Enabled native filtering so the client can search within any column
        filter_action='native',
        # Set row_selectable to single as required for the map callback to function
        row_selectable='single',
        # Selected first row by default so the map always has a location to display
        selected_rows=[0],
        # Styled header row with dark blue background for readability
        style_header={
            'backgroundColor': '#003366',
            'color': 'white',
            'fontWeight': 'bold'
        },
        # Enabled horizontal scrolling so all columns are accessible
        style_table={'overflowX': 'auto'},
    ),

    html.Br(),
    html.Hr(),

    # This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',
            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    # Added filter logic to query MongoDB using the CRUD module
    # based on the rescue type selected by the user
    # Each filter uses the breed list, sex, and age range from the Dashboard Specifications Document

    # Water Rescue filter: Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
    # Preferred sex: Intact Female | Age range: 26 to 156 weeks
    if filter_type == 'Water':
        dff = pd.DataFrame.from_records(
            shelter.read({
                "animal_type": "Dog",
                "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
                "sex_upon_outcome": "Intact Female",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        )
    # Mountain or Wilderness Rescue filter: German Shepherd, Alaskan Malamute,
    # Old English Sheepdog, Siberian Husky, Rottweiler
    # Preferred sex: Intact Male | Age range: 26 to 156 weeks
    elif filter_type == 'Wilderness':
        dff = pd.DataFrame.from_records(
            shelter.read({
                "animal_type": "Dog",
                "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
            })
        )
    # Disaster or Individual Tracking filter: Doberman Pinscher, German Shepherd,
    # Golden Retriever, Bloodhound, Rottweiler
    # Preferred sex: Intact Male | Age range: 20 to 300 weeks
    elif filter_type == 'Disaster':
        dff = pd.DataFrame.from_records(
            shelter.read({
                "animal_type": "Dog",
                "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
                "sex_upon_outcome": "Intact Male",
                "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
            })
        )
    # Reset filter: returns all records with no filter applied
    else:
        dff = pd.DataFrame.from_records(shelter.read({}))

    # Drop the _id column from filtered results for the same reason as the initial load above
    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    # Added a pie chart showing breed distribution of the currently filtered animals
    # This serves as the second required chart beyond the geolocation map
    # Limited to top 10 breeds to keep the chart readable for the client
    dff = pd.DataFrame.from_dict(viewData)
    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    breed_counts.columns = ['breed', 'count']

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                values='count',
                names='breed',
                title='Breed Distribution of Filtered Animals'
            )
        )
    ]


# This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Added a check for None to prevent a callback error on initial page load
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]

# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None:
        return
    elif index is None:
        return

    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else:
        row = index[0]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75, -97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row, 13], dff.iloc[row, 14]], children=[
                dl.Tooltip(dff.iloc[row, 4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row, 9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app,
# the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server()

Dash app running on https://logobattery-monkeystick-3000.codio.io/proxy/8050/
